# Laboratório: Baixando arquivos da internet com Python (`requests`) 🌐⬇️

**Disciplina:** Laboratório de Programação — Ciência da Computação (UFPI)  
**Prof.:** Armando Soares Sousa — Departamento de Computação

---

## 🎯 Objetivos da atividade

Ao final deste laboratório você será capaz de:

1. Explicar, de forma simples, como funciona uma **requisição HTTP** (cliente ↔ servidor).
2. Identificar as partes de uma **URL**.
3. Usar a biblioteca **`requests`** para buscar páginas, textos, dados JSON e arquivos.
4. Interpretar uma **resposta**: código de status, cabeçalhos e conteúdo.
5. **Tratar erros** de rede (endereço inválido, arquivo inexistente, tempo esgotado).
6. Salvar **arquivos binários** (imagens, áudio, vídeo) no disco.
7. Baixar arquivos grandes **em pedaços** (*streaming*) com uma **barra de progresso**.
8. Organizar o código em **funções** e em uma **classe** reutilizável.

## 🗺️ Roteiro sugerido (≈ 100 minutos)

| Parte | Conteúdo | Tempo |
|---|---|---|
| 0 | Preparação do ambiente | 5 min |
| 1 | Como a web funciona: cliente, servidor e URL | 10 min |
| 2 | Primeiras requisições com `requests` | 20 min |
| 3 | Quando algo dá errado: erros e exceções | 10 min |
| 4 | Texto × bytes: salvando um arquivo | 10 min |
| 5 | Download em pedaços (*streaming*) | 10 min |
| 6 | Barra de progresso com `tqdm` | 10 min |
| 7 | Organizando o código: funções e classe | 15 min |
| 8 | Exercícios | 10+ min |

## 🧭 Como usar este notebook

- Execute as células **na ordem** (`Shift + Enter`).
- **Leia o texto antes de executar** e tente **prever o resultado**.
- Blocos **🤔 Pare e pense** são para discussão com a turma.
- Blocos **✍️ Sua vez** têm células vazias para você escrever código.
- Este notebook **precisa de internet**: ele vai baixar arquivos de verdade.

---
# Parte 0 — Preparação do ambiente

| Biblioteca | Para que serve | Já vem com o Python? |
|---|---|---|
| `requests` | fazer requisições HTTP (buscar coisas na internet) | não, mas já vem no Colab |
| `tqdm` | mostrar barras de progresso | não, mas já vem no Colab |
| `urllib.parse` | separar as partes de uma URL | sim |
| `os` | lidar com arquivos e caminhos | sim |

Fora do Colab, instale com:
```
pip install requests tqdm
```

In [ ]:
# Se estiver fora do Colab e faltar alguma biblioteca, descomente a linha abaixo:
# !pip install requests tqdm

import os
from urllib.parse import urlparse

import requests
from tqdm.auto import tqdm   # escolhe automaticamente a melhor barra (notebook ou terminal)

print("Versão do requests:", requests.__version__)

Vamos guardar em variáveis as URLs que usaremos ao longo do notebook:

In [ ]:
BASE = "https://raw.githubusercontent.com/armandossrecife/teste/main/"

url_imagem = BASE + "babydogcat.png"
url_foto   = BASE + "screen_matrix.jpeg"
url_audio  = BASE + "Kalimba.mp3"
url_video  = BASE + "Adrienne.mp4"
url_texto  = BASE + "README.md"

print(url_imagem)

---
# Parte 1 — Como a web funciona: cliente, servidor e URL

## 1.1 Requisição e resposta

Toda vez que você abre uma página ou baixa um arquivo, acontece uma "conversa" entre dois programas:

- o **cliente** (seu navegador, ou o seu programa em Python) faz um **pedido** — a **requisição** (*request*);
- o **servidor** (um computador na internet) devolve uma **resposta** (*response*).

```
   CLIENTE (seu Python)                             SERVIDOR (github)
  ┌───────────────────┐   1. requisição HTTP      ┌──────────────────┐
  │ requests.get(url) │ ────────────────────────▶ │                  │
  │                   │   GET /babydogcat.png     │  procura o       │
  │                   │                           │  arquivo...      │
  │                   │   2. resposta HTTP        │                  │
  │  response         │ ◀──────────────────────── │                  │
  │  .status_code=200 │   status + cabeçalhos     │                  │
  │  .headers         │   + conteúdo (bytes)      │                  │
  │  .content         │                           │                  │
  └───────────────────┘                           └──────────────────┘
```

A "língua" dessa conversa se chama **HTTP** (*HyperText Transfer Protocol*). O **HTTPS** é a mesma coisa, só que **criptografada**.

### Os métodos HTTP mais comuns

| Método | Significado | Em `requests` |
|---|---|---|
| `GET` | **buscar** algo (página, arquivo, dados) | `requests.get(url)` |
| `POST` | **enviar** dados (formulário, cadastro) | `requests.post(url, data=...)` |
| `HEAD` | buscar **só os cabeçalhos**, sem o conteúdo | `requests.head(url)` |
| `PUT` / `DELETE` | atualizar / apagar um recurso | `requests.put(...)` / `requests.delete(...)` |

Neste laboratório usaremos principalmente **`GET`** (e um pouco de `HEAD`).

## 1.2 Anatomia de uma URL

Uma **URL** é o "endereço" de um recurso na internet:

```
https://raw.githubusercontent.com/armandossrecife/teste/main/babydogcat.png?v=2
└─┬─┘   └─────────┬─────────────┘└──────────────┬──────────────────────────┘└┬┘
esquema     domínio (host)                caminho (path)                  consulta
(scheme)                                                                  (query)
```

A função `urlparse` separa essas partes para nós:

In [ ]:
partes = urlparse("https://raw.githubusercontent.com/armandossrecife/teste/main/babydogcat.png?v=2")

print("esquema :", partes.scheme)
print("domínio :", partes.netloc)
print("caminho :", partes.path)
print("consulta:", partes.query)

O **nome do arquivo** é a última parte do caminho. O módulo `os.path` sabe extraí-lo e separar a extensão:

In [ ]:
caminho = partes.path
nome_completo = os.path.basename(caminho)           # última parte do caminho
nome, extensao = os.path.splitext(nome_completo)    # separa nome e extensão

print("nome completo:", nome_completo)
print("nome         :", nome)
print("extensão     :", extensao)

> 🤔 **Pare e pense:** por que usar `os.path.splitext` em vez de `nome_completo.split(".")`?
> Teste os dois com o nome `"dados.tar.gz"` e com `"LEIAME"` (sem extensão) na célula abaixo.

In [ ]:
for exemplo in ["babydogcat.png", "dados.tar.gz", "LEIAME"]:
    print(f"{exemplo:<16} split('.') → {exemplo.split('.')!s:<24} splitext → {os.path.splitext(exemplo)}")

Com `split(".")`, pegar `[1]` daria a "extensão" errada para `dados.tar.gz` e um **erro** (`IndexError`) para `LEIAME`.
O `splitext` lida bem com os dois casos.

---
# Parte 2 — Primeiras requisições com `requests`

## 2.1 Buscando um texto

Vamos pedir ao servidor o arquivo `README.md`, que é um texto simples:

In [ ]:
resposta = requests.get(url_texto, timeout=10)

print("Tipo do objeto:", type(resposta))
print("Código de status:", resposta.status_code)

> 💡 **Sempre use `timeout`.** Ele diz quantos segundos esperar pelo servidor. Sem ele, se o servidor não responder, seu programa pode ficar **parado para sempre**.

## 2.2 O código de status

O **código de status** é um número de 3 dígitos que resume o que aconteceu. O primeiro dígito diz a "família":

| Família | Significado | Exemplos |
|---|---|---|
| **2xx** | ✅ deu certo | `200 OK`, `201 Created` |
| **3xx** | ↪️ redirecionamento (o recurso está em outro lugar) | `301 Moved Permanently`, `304 Not Modified` |
| **4xx** | ❌ erro do **cliente** (pedido errado) | `400 Bad Request`, `403 Forbidden`, `404 Not Found`, `429 Too Many Requests` |
| **5xx** | 💥 erro do **servidor** | `500 Internal Server Error`, `503 Service Unavailable` |

In [ ]:
print("status_code:", resposta.status_code)   # número
print("reason     :", resposta.reason)        # texto que acompanha o número
print("ok         :", resposta.ok)            # True se o status for menor que 400
print("url final  :", resposta.url)           # endereço que realmente respondeu

## 2.3 O conteúdo da resposta: `.text` e `.content`

A resposta traz o conteúdo de duas formas:

| Atributo | Tipo | Quando usar |
|---|---|---|
| `resposta.text` | `str` (texto) | páginas HTML, CSV, Markdown, JSON... |
| `resposta.content` | `bytes` (dados "crus") | imagens, áudio, vídeo, PDF, ZIP... **qualquer arquivo** |

In [ ]:
print(type(resposta.text))
print(resposta.text[:300])   # os primeiros 300 caracteres

In [ ]:
print(type(resposta.content))
print(resposta.content[:60])  # os primeiros 60 bytes

Repare no `b'...'` no início: é assim que o Python mostra um objeto do tipo **`bytes`**.

## 2.4 Os cabeçalhos (*headers*)

Antes do conteúdo, o servidor envia **cabeçalhos**: informações **sobre** a resposta (metadados).

In [ ]:
for chave, valor in resposta.headers.items():
    print(f"{chave:<28} {valor}")

Os cabeçalhos mais úteis para nós:

| Cabeçalho | Significado |
|---|---|
| `Content-Type` | o **tipo** do conteúdo (ex.: `image/png`, `text/plain`, `audio/mpeg`) |
| `Content-Length` | o **tamanho** do conteúdo, em **bytes** |
| `Date` | data e hora da resposta |
| `Cache-Control` | por quanto tempo a resposta pode ser guardada em cache |

O objeto `headers` se comporta como um **dicionário** — e não diferencia maiúsculas de minúsculas:

In [ ]:
print(resposta.headers["Content-Type"])
print(resposta.headers["content-type"])            # mesma coisa!
print(resposta.headers.get("X-Nao-Existe", "—"))  # .get com valor padrão, sem erro

## 2.5 Perguntando o tamanho antes de baixar: `HEAD`

O método `HEAD` pede **só os cabeçalhos**. É útil para saber o tipo e o tamanho de um arquivo **sem baixá-lo**:

In [ ]:
for url in [url_imagem, url_foto, url_audio, url_video]:
    r = requests.head(url, timeout=10)
    tamanho = int(r.headers.get("Content-Length", 0))
    tipo = r.headers.get("Content-Type")
    nome = os.path.basename(urlparse(url).path)
    print(f"{nome:<20} {tipo:<26} {tamanho / 1024 / 1024:6.2f} MB")

> 💡 Dividimos por 1024 duas vezes: **bytes → KB → MB**.

## 2.6 Recebendo dados estruturados: JSON

Muitos serviços na internet (as **APIs**) respondem em **JSON**, um formato de texto que lembra dicionários e listas do Python.
O método `.json()` converte o texto em objetos Python:

In [ ]:
url_json = "https://raw.githubusercontent.com/dariusk/corpora/master/data/animals/dogs.json"

r = requests.get(url_json, timeout=10)
dados = r.json()           # texto JSON  →  dicionário Python

print(type(dados))
print("Chaves:", list(dados.keys()))
print("Descrição:", dados["description"])
print("Total de raças:", len(dados["dogs"]))
print("Primeiras 5:", dados["dogs"][:5])

## 2.7 Enviando parâmetros na URL: `params`

Muitas URLs aceitam **parâmetros de consulta** depois do `?` (ex.: uma busca).
Em vez de montar o texto à mão, passamos um **dicionário** em `params` e o `requests` monta a URL — inclusive tratando espaços e acentos.

Para ver a URL montada **sem fazer a requisição**, podemos "preparar" o pedido:

In [ ]:
pedido = requests.Request(
    "GET",
    "https://www.google.com/search",
    params={"q": "pandas python", "hl": "pt-BR"},
).prepare()

print(pedido.url)

> 💡 O espaço virou `+`. Na prática, você escreveria simplesmente `requests.get(url, params={...}, timeout=10)`.

### ✍️ Sua vez

1. Faça um `GET` em `url_foto` e mostre o `status_code`, o `Content-Type` e o tamanho em bytes do conteúdo (`len(resposta.content)`).
2. Confira se o tamanho é igual ao informado no cabeçalho `Content-Length`.

In [ ]:
# Sua vez

---
# Parte 3 — Quando algo dá errado: erros e exceções

Em programas que usam a rede, **erros são normais**: o arquivo pode não existir, a internet pode cair, o endereço pode estar errado.
Um bom programa **prevê** essas situações.

## 3.1 O servidor respondeu, mas com erro (404)

Importante: um **404 não gera exceção** automaticamente. A requisição "funcionou" — o servidor respondeu dizendo que o arquivo não existe.

In [ ]:
url_inexistente = BASE + "nao_existe.png"

r = requests.get(url_inexistente, timeout=10)
print(r.status_code, r.reason, "| ok =", r.ok)
print("Conteúdo:", r.text)

> 🤔 **Pare e pense:** o que aconteceria se salvássemos `r.content` em um arquivo `nao_existe.png`? (Resposta: teríamos um "PNG" que, na verdade, contém o texto `404: Not Found` — um arquivo corrompido!)

Para transformar um status de erro (4xx ou 5xx) em **exceção**, usamos `raise_for_status()`:

In [ ]:
try:
    r = requests.get(url_inexistente, timeout=10)
    r.raise_for_status()          # lança HTTPError se o status for 4xx ou 5xx
    print("Tudo certo!")
except requests.exceptions.HTTPError as erro:
    print("Erro HTTP:", erro)

## 3.2 As principais exceções do `requests`

Todas as exceções do `requests` são "filhas" de `RequestException`:

```
RequestException                     ← captura qualquer erro do requests
 ├── HTTPError          status 4xx/5xx (via raise_for_status)
 ├── ConnectionError    não conseguiu conectar (sem internet, domínio inexistente)
 ├── Timeout            o servidor demorou mais que o timeout
 ├── MissingSchema      URL sem "http://" ou "https://"
 └── InvalidSchema      esquema que o requests não sabe usar (ex.: ftp://)
```

Vamos provocar cada erro de propósito:

In [ ]:
testes = [
    ("URL sem esquema",        "qualquer coisa",                                  10),
    ("Esquema não suportado",  "ftp://ftp.exemplo.com/arquivo.txt",               10),
    ("Domínio inexistente",    "https://dominio-que-nao-existe-123456.com.br",    10),
    ("Tempo esgotado",         url_video,                                         0.0001),
]

for descricao, url, limite in testes:
    try:
        requests.get(url, timeout=limite)
        print(f"{descricao:<24} → sem erro")
    except requests.exceptions.RequestException as erro:
        print(f"{descricao:<24} → {type(erro).__name__}")

> 💡 Repare que o `requests` **não** baixa arquivos por **FTP** — só HTTP e HTTPS. Para FTP existe o módulo `ftplib` do Python.

> 💡 Ordem dos `except`: coloque as exceções **mais específicas primeiro** e a mais geral (`RequestException`) por último. O Python usa o **primeiro** `except` compatível.

```python
try:
    ...
except requests.exceptions.Timeout:
    print("O servidor demorou demais.")
except requests.exceptions.ConnectionError:
    print("Sem conexão. Verifique a internet.")
except requests.exceptions.RequestException as erro:
    print("Outro erro:", erro)
```

---
# Parte 4 — Texto × bytes: salvando um arquivo

## 4.1 Todo arquivo é uma sequência de bytes

Uma imagem, uma música ou um vídeo são, no fundo, **sequências de bytes** (números de 0 a 255).
Por isso, para salvá-los, abrimos o arquivo em **modo binário**:

| Modo | Significado | Escrevemos... |
|---|---|---|
| `"w"` | escrita de **texto** | `str` |
| `"wb"` | escrita **binária** (*write binary*) | `bytes` |
| `"r"` / `"rb"` | leitura de texto / binária | — |

## 4.2 O jeito mais simples de baixar um arquivo

In [ ]:
r = requests.get(url_imagem, timeout=30)
r.raise_for_status()

with open("imagem_simples.png", "wb") as arquivo:   # 'wb' = escrita binária
    arquivo.write(r.content)

print("Arquivo salvo! Tamanho:", os.path.getsize("imagem_simples.png"), "bytes")

> 💡 O `with open(...) as arquivo:` garante que o arquivo será **fechado** ao final do bloco, mesmo se ocorrer um erro.

Vamos conferir se a imagem está correta, exibindo-a no notebook:

In [ ]:
from IPython.display import Image, display

display(Image(filename="imagem_simples.png", width=300))

## 4.3 Curiosidade: a "assinatura" dos arquivos 🔍

Muitos formatos começam com alguns bytes fixos, chamados **números mágicos** (*magic numbers*). É assim que os programas reconhecem o tipo de um arquivo, mesmo sem olhar a extensão.

| Formato | Primeiros bytes |
|---|---|
| PNG | `89 50 4E 47` (`\x89PNG`) |
| JPEG | `FF D8 FF` |
| MP3 (com etiqueta ID3) | `49 44 33` (`ID3`) |

Vamos conferir lendo só o começo de cada arquivo. Com `stream=True` (veremos a seguir) **não precisamos baixar o arquivo inteiro** para isso:

In [ ]:
for url in [url_imagem, url_foto, url_audio]:
    with requests.get(url, stream=True, timeout=10) as r:
        primeiros = next(r.iter_content(chunk_size=8))   # só os 8 primeiros bytes
    nome = os.path.basename(urlparse(url).path)
    print(f"{nome:<20} {primeiros.hex(' ').upper():<26} {primeiros}")

---
# Parte 5 — Download em pedaços (*streaming*)

## 5.1 O problema do `r.content`

`r.content` guarda **o arquivo inteiro na memória RAM** antes de salvar.
Para uma imagem de 2 MB, tudo bem. Mas e um vídeo de 4 GB?

**Analogia:** encher uma piscina com um **balde gigante** (você precisa carregar toda a água de uma vez) × com uma **mangueira** (a água passa aos poucos, em fluxo contínuo).

```
SEM streaming (r.content)              COM streaming (iter_content)
servidor ──▶ [ arquivo inteiro ] ──▶ disco      servidor ──▶ [pedaço] ──▶ disco
             na memória RAM                                ──▶ [pedaço] ──▶ disco
                                                           ──▶ [pedaço] ──▶ disco
memória usada ≈ tamanho do arquivo      memória usada ≈ tamanho de um pedaço
```

## 5.2 Como fazer: `stream=True` + `iter_content`

1. `stream=True` diz ao `requests`: "não baixe o conteúdo ainda; eu vou ler aos poucos".
2. `iter_content(chunk_size=...)` entrega o conteúdo em **pedaços** (*chunks*) de tamanho fixo.
3. Cada pedaço é escrito no arquivo assim que chega.

Primeiro, vamos só **contar** os pedaços para entender o mecanismo:

In [ ]:
TAMANHO_PEDACO = 64 * 1024   # 64 KB por pedaço

with requests.get(url_imagem, stream=True, timeout=30) as r:
    r.raise_for_status()
    total = int(r.headers.get("Content-Length", 0))
    print(f"Tamanho anunciado: {total:,} bytes")

    qtd_pedacos = 0
    recebidos = 0
    for pedaco in r.iter_content(chunk_size=TAMANHO_PEDACO):
        qtd_pedacos += 1
        recebidos += len(pedaco)
        if qtd_pedacos <= 3:
            print(f"pedaço {qtd_pedacos}: {len(pedaco):,} bytes")

print("...")
print(f"Total de pedaços: {qtd_pedacos}  |  bytes recebidos: {recebidos:,}")

> 🤔 **Pare e pense:** $2.209.621 \div 65.536 \approx 33{,}7$. Por que foram **34** pedaços? Qual o tamanho do último?

> 💡 `iter_content` é um **gerador**: ele não guarda todos os pedaços; produz um de cada vez, à medida que o `for` pede.

Agora sim, gravando no disco:

In [ ]:
with requests.get(url_imagem, stream=True, timeout=30) as r:
    r.raise_for_status()
    with open("imagem_streaming.png", "wb") as arquivo:
        for pedaco in r.iter_content(chunk_size=TAMANHO_PEDACO):
            arquivo.write(pedaco)

print("Tamanho salvo:", os.path.getsize("imagem_streaming.png"), "bytes")

## 5.3 Cuidado com uma armadilha comum ⚠️

Se você usar `stream=True` mas acessar `r.content` (ou `len(r.content)`), o `requests` **baixa tudo para a memória** naquele momento — e o *streaming* perde o sentido.

```python
r = requests.get(url, stream=True)
len(r.content)                      # ❌ baixou tudo para a memória aqui!
for pedaco in r.iter_content(...):  #    os "pedaços" agora vêm da memória, não da rede
    ...
```

Para saber o tamanho, use o **cabeçalho** `Content-Length`, que chega antes do conteúdo.

> 💡 Usar `with requests.get(...) as r:` garante que a **conexão será fechada** ao final, liberando recursos.

---
# Parte 6 — Barra de progresso com `tqdm`

Quando um download demora, é bom mostrar ao usuário **quanto já foi feito**.
A biblioteca **`tqdm`** (https://tqdm.github.io) desenha barras de progresso com pouquíssimo código.

## 6.1 Forma 1: envolvendo um laço

Basta "envolver" qualquer iterável com `tqdm(...)`:

In [ ]:
import time

for i in tqdm(range(20), desc="Processando"):
    time.sleep(0.1)    # simula um trabalho de 0,1 s

## 6.2 Forma 2: atualizando manualmente

Quando não temos um iterável com tamanho conhecido (como um download), criamos a barra com um **total** e chamamos `update(n)` a cada avanço:

In [ ]:
total_passos = 10

with tqdm(total=total_passos, desc="Simulando", unit="passo") as barra:
    for _ in range(total_passos):
        time.sleep(0.2)
        barra.update(1)     # avança 1 unidade

Parâmetros úteis do `tqdm`:

| Parâmetro | Para que serve |
|---|---|
| `total` | quanto é 100% |
| `desc` | texto à esquerda da barra |
| `unit` | unidade mostrada (`"B"` para bytes) |
| `unit_scale=True` | converte automaticamente para k, M, G (KB, MB, GB) |
| `unit_divisor=1024` | usa 1024 (em vez de 1000) para a conversão |

## 6.3 Juntando tudo: download com barra de progresso

A ideia: o **total** da barra é o `Content-Length`, e a cada pedaço recebido avançamos `len(pedaco)` bytes.

In [ ]:
with requests.get(url_audio, stream=True, timeout=30) as r:
    r.raise_for_status()
    total = int(r.headers.get("Content-Length", 0))

    with open("Kalimba.mp3", "wb") as arquivo, \
         tqdm(total=total, unit="B", unit_scale=True, unit_divisor=1024, desc="Kalimba.mp3") as barra:
        for pedaco in r.iter_content(chunk_size=TAMANHO_PEDACO):
            arquivo.write(pedaco)
            barra.update(len(pedaco))

print("Tamanho salvo:", os.path.getsize("Kalimba.mp3"), "bytes")

> 🤔 **Pare e pense:** nem todo servidor envia o `Content-Length`. Nesse caso, `total` seria `0`. O que acontece com a barra?
> (O `tqdm` aceita `total=None`: a barra mostra quanto foi baixado e a velocidade, mas sem a porcentagem.)

Quer ouvir o áudio baixado? Descomente a célula abaixo (o áudio fica embutido no notebook e o arquivo `.ipynb` cresce ~8 MB):

In [ ]:
# from IPython.display import Audio
# Audio("Kalimba.mp3")

---
# Parte 7 — Organizando o código: funções e classe

Até aqui repetimos o mesmo código várias vezes. Vamos **encapsulá-lo** para reutilizar.

## 7.1 Função para extrair o nome do arquivo da URL

In [ ]:
ESQUEMAS_SUPORTADOS = ("http", "https")   # o requests não faz download via FTP

def nome_arquivo_da_url(url):
    # Devolve o nome do arquivo presente no final da URL (ex.: 'babydogcat.png').
    partes = urlparse(url)

    if partes.scheme not in ESQUEMAS_SUPORTADOS:
        raise ValueError(f"Esquema não suportado: '{partes.scheme}' (use http ou https)")

    nome = os.path.basename(partes.path)
    if not nome:
        raise ValueError("A URL não indica um nome de arquivo")
    return nome

print(nome_arquivo_da_url(url_imagem))
print(nome_arquivo_da_url(url_video))

In [ ]:
# Testando os casos de erro
for url in ["qualquer coisa", "ftp://servidor/arquivo.txt", "https://github.com/"]:
    try:
        nome_arquivo_da_url(url)
    except ValueError as erro:
        print(f"{url:<30} → {erro}")

## 7.2 Função de download

A função abaixo reúne tudo o que aprendemos: `timeout`, `raise_for_status`, *streaming*, gravação binária e barra de progresso.
Ela tem **parâmetros com valores padrão**, então pode ser chamada de forma simples ou personalizada.

In [ ]:
def baixar_arquivo(url, destino=None, tamanho_pedaco=64 * 1024, mostrar_progresso=True):
    # Baixa o arquivo da URL e salva em 'destino'. Devolve o caminho do arquivo salvo.
    if destino is None:
        destino = nome_arquivo_da_url(url)    # usa o nome que está na URL

    with requests.get(url, stream=True, timeout=30) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0)) or None   # 0 vira None

        with open(destino, "wb") as arquivo, \
             tqdm(total=total, unit="B", unit_scale=True, unit_divisor=1024,
                  desc=destino, disable=not mostrar_progresso) as barra:
            for pedaco in r.iter_content(chunk_size=tamanho_pedaco):
                arquivo.write(pedaco)
                barra.update(len(pedaco))

    return destino

In [ ]:
caminho = baixar_arquivo(url_foto)
print("Salvo em:", caminho, "|", os.path.getsize(caminho), "bytes")

> 💡 `int(...) or None`: se o resultado for `0` (falso), a expressão vale `None`. É um "atalho" comum em Python.

E se der errado? A função **não esconde** o erro — ela deixa a exceção "subir" para quem a chamou decidir o que fazer:

In [ ]:
try:
    baixar_arquivo(url_inexistente)
except requests.exceptions.HTTPError as erro:
    print("Não foi possível baixar:", erro)

> 🤔 **Pare e pense:** a chamada acima deixou um arquivo `nao_existe.png` no disco? Por quê?
> (Dica: em que linha a exceção acontece? Antes ou depois do `open`?)

## 7.3 Baixando vários arquivos

In [ ]:
urls = [url_imagem, url_foto, url_audio, url_video]

baixados = []
for url in urls:
    try:
        baixados.append(baixar_arquivo(url))
    except (requests.exceptions.RequestException, ValueError) as erro:
        print(f"Falha em {url}: {erro}")

print("\nArquivos baixados:")
for nome in baixados:
    print(f"  {nome:<20} {os.path.getsize(nome) / 1024 / 1024:6.2f} MB")

## 7.4 De função para classe (com *callback*)

Uma **classe** agrupa **dados** (a URL, o destino) e **comportamentos** (executar o download).
Vamos acrescentar uma ideia nova: o ***callback***.

Um *callback* é uma **função que você entrega para outra função chamar depois**. Aqui, a classe chamará o *callback* a cada pedaço baixado, informando o progresso.
Isso permite, por exemplo, atualizar uma interface gráfica, gravar um log ou avisar em certos marcos — **sem mudar o código da classe**.

In [ ]:
class Download:
    # Baixa um arquivo da internet, com barra de progresso e callback opcional.

    def __init__(self, url, destino=None, tamanho_pedaco=64 * 1024):
        self.url = url
        self.destino = destino or nome_arquivo_da_url(url)
        self.tamanho_pedaco = tamanho_pedaco
        self.callback = None          # função opcional: callback(baixados, total)

    def set_callback(self, funcao):
        self.callback = funcao

    def executar(self):
        with requests.get(self.url, stream=True, timeout=30) as r:
            r.raise_for_status()
            total = int(r.headers.get("Content-Length", 0))
            baixados = 0

            with open(self.destino, "wb") as arquivo, \
                 tqdm(total=total or None, unit="B", unit_scale=True,
                      unit_divisor=1024, desc=self.destino) as barra:
                for pedaco in r.iter_content(chunk_size=self.tamanho_pedaco):
                    arquivo.write(pedaco)
                    baixados += len(pedaco)
                    barra.update(len(pedaco))
                    if self.callback:
                        self.callback(baixados, total)

        return self.destino

Agora criamos um *callback* que avisa quando o download passa por 25%, 50%, 75% e 100%:

In [ ]:
marcos_avisados = set()

def avisar_marcos(baixados, total):
    if not total:
        return
    porcentagem = baixados * 100 // total
    for marco in (25, 50, 75, 100):
        if porcentagem >= marco and marco not in marcos_avisados:
            marcos_avisados.add(marco)
            print(f"  ✔ {marco}% concluído")

d = Download(url_imagem, "imagem_classe.png")
d.set_callback(avisar_marcos)
d.executar()

> 🤔 **Pare e pense:** a classe `Download` não sabe **nada** sobre "marcos de 25%". Quem decide o que fazer com o progresso é o *callback*. Que outros *callbacks* você poderia criar?

---
# Parte 8 — ✍️ Exercícios

**Exercício 1.** Faça um `HEAD` em cada URL da lista `urls` e mostre uma tabela com: nome do arquivo, `Content-Type` e tamanho em KB.

In [ ]:
# Exercício 1

**Exercício 2.** Baixe o arquivo JSON `https://raw.githubusercontent.com/dariusk/corpora/master/data/foods/fruits.json` e responda: quantas frutas existem? Quais começam com a letra **"b"**?

In [ ]:
# Exercício 2

**Exercício 3.** Modifique `baixar_arquivo` para que, quando o destino **já existir**, ela **não baixe de novo** e apenas avise o usuário.

*Dica:* `os.path.exists(caminho)`.

In [ ]:
# Exercício 3

**Exercício 4.** Crie uma pasta `downloads` e faça `baixar_arquivo` salvar os arquivos lá dentro.

*Dica:* `os.makedirs("downloads", exist_ok=True)` e `os.path.join("downloads", nome)`.

In [ ]:
# Exercício 4

**Exercício 5.** Meça o **tempo** de download do vídeo usando `tamanho_pedaco` de 1 KB, 64 KB e 1 MB. Há diferença? Por quê?

*Dica:* `inicio = time.perf_counter()` antes e `time.perf_counter() - inicio` depois.

In [ ]:
# Exercício 5

**Exercício 6.** Crie um *callback* que guarde em uma lista o percentual a cada chamada e, no final, mostre quantas vezes ele foi chamado. Relacione esse número com o tamanho do pedaço.

In [ ]:
# Exercício 6

**Exercício 7.** Depois de baixar, confira se o arquivo está **completo**: compare `os.path.getsize(destino)` com o `Content-Length`. Se forem diferentes, lance uma exceção e apague o arquivo incompleto (`os.remove`).

In [ ]:
# Exercício 7

**Exercício 8 (desafio).** Escreva uma função `identificar_tipo(caminho)` que leia os primeiros bytes de um arquivo **local** (`open(caminho, "rb").read(8)`) e diga se ele é PNG, JPEG ou MP3, usando a tabela de números mágicos da Parte 4.3. Teste com os arquivos baixados.

In [ ]:
# Exercício 8

**Exercício 9 (desafio, com moderação).** A API pública do GitHub responde em JSON. Busque `https://api.github.com/repos/armandossrecife/teste` e mostre o nome, a descrição e a data de criação do repositório.

⚠️ Sem autenticação, a API do GitHub aceita poucas requisições por hora **por endereço IP**. Em um laboratório, todos os computadores podem compartilhar o mesmo IP — se aparecer o status **403** ou **429**, você encontrou o limite! Trate esse caso no seu código.

In [ ]:
# Exercício 9

---
# 📌 Resumo — "Cola" dos comandos

| Objetivo | Código |
|---|---|
| Buscar um recurso | `r = requests.get(url, timeout=10)` |
| Só os cabeçalhos | `requests.head(url, timeout=10)` |
| Enviar parâmetros | `requests.get(url, params={"q": "x"}, timeout=10)` |
| Código de status | `r.status_code`, `r.reason`, `r.ok` |
| Lançar exceção se 4xx/5xx | `r.raise_for_status()` |
| Conteúdo como texto | `r.text` |
| Conteúdo como bytes | `r.content` |
| Conteúdo JSON | `r.json()` |
| Cabeçalhos | `r.headers["Content-Type"]`, `r.headers.get("Content-Length")` |
| Download em pedaços | `with requests.get(url, stream=True, timeout=30) as r:` + `r.iter_content(chunk_size=...)` |
| Gravar bytes | `with open(nome, "wb") as f: f.write(dados)` |
| Partes da URL | `urlparse(url).scheme / .netloc / .path / .query` |
| Nome e extensão | `os.path.basename(caminho)`, `os.path.splitext(nome)` |
| Tamanho de arquivo local | `os.path.getsize(caminho)` |
| Barra de progresso | `tqdm(total=..., unit="B", unit_scale=True)` + `barra.update(n)` |
| Capturar erros | `except requests.exceptions.RequestException as erro:` |

## 🎓 Boas práticas

1. **Sempre** use `timeout` nas requisições.
2. **Sempre** verifique o status (`raise_for_status()`) antes de salvar o conteúdo.
3. Para arquivos grandes, use **`stream=True` + `iter_content`** e **não** acesse `r.content`.
4. Use `with` para fechar **arquivos** e **conexões** automaticamente.
5. Abra arquivos binários com **`"wb"`**.
6. Trate as exceções de rede: elas **vão** acontecer.
7. Seja educado com os servidores: não faça requisições em excesso.

## 📚 Referências

- Documentação do `requests`: https://requests.readthedocs.io/
- Guia rápido (*Quickstart*): https://requests.readthedocs.io/en/latest/user/quickstart/
- Códigos de status HTTP (MDN): https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Status
- `urllib.parse`: https://docs.python.org/pt-br/3/library/urllib.parse.html
- `tqdm`: https://tqdm.github.io